In [1]:
print("starting")

starting


In [3]:
import os
import h5py
import numpy as np
from sklearn.decomposition import PCA
from tqdm import tqdm

# --- 1. Paths & Configuration ---
output_dir = "/media/chris-remote/Projects/ONeill/estimation/output/kevin_2cylinder"
unified_h5 = os.path.join(output_dir, "flipflop_velocity_unified.h5")
save_path = os.path.join(output_dir, "lstm_ready_data_FF_nearwake.h5")

# NEW SENSOR LOCATION
SENSOR_X = 5.0 
TOLERANCE = 0.1 
N_MODES_FF = 65    # Sticking to the 65 converged modes
N_MODES_SS = 100   # Sufficient for the near-wake strip

# --- 2. Load and Align Grid ---
print(f"Loading data and extracting strip at x={SENSOR_X}...")
with h5py.File(unified_h5, 'r') as f:
    n_data_pts = f['train/ux'].shape[1]
    x = f['x'][:n_data_pts]
    y = f['y'][:n_data_pts]
    ux_train = f['train/ux'][:]
    uy_train = f['train/uy'][:]
    ux_test = f['test/ux'][:]
    uy_test = f['test/uy'][:]

# Find Sensor Indices
sensor_mask = np.abs(x - SENSOR_X) < TOLERANCE
sensor_idx = np.where(sensor_mask)[0]
sensor_idx = sensor_idx[np.argsort(y[sensor_idx])] 

print(f"Extracted {len(sensor_idx)} points at x={SENSOR_X}.")

# --- 3. Full-Field & Sensor POD ---
print("Computing PODs...")
S_train = np.hstack([ux_train, uy_train])
mean_ff = np.mean(S_train, axis=0)
pca_ff = PCA(n_components=N_MODES_FF, svd_solver='randomized', random_state=42)
a_ff_train = pca_ff.fit_transform(S_train - mean_ff)

ss_train = np.hstack([ux_train[:, sensor_idx], uy_train[:, sensor_idx]])
mean_ss = np.mean(ss_train, axis=0)
pca_ss = PCA(n_components=N_MODES_SS, svd_solver='randomized', random_state=42)
a_ss_train = pca_ss.fit_transform(ss_train - mean_ss)

# --- 4. Projection & Save ---
a_ff_test = pca_ff.transform(np.hstack([ux_test, uy_test]) - mean_ff)
a_ss_test = pca_ss.transform(np.hstack([ux_test[:, sensor_idx], uy_test[:, sensor_idx]]) - mean_ss)

with h5py.File(save_path, 'w') as f:
    f.create_dataset('phi_ff', data=pca_ff.components_)
    f.create_dataset('mean_ff', data=mean_ff)
    f.create_dataset('phi_ss', data=pca_ss.components_)
    f.create_dataset('mean_ss', data=mean_ss)
    f.create_dataset('sensor_idx', data=sensor_idx)
    tr = f.create_group('train')
    tr.create_dataset('a_ff', data=a_ff_train)
    tr.create_dataset('a_ss', data=a_ss_train)
    ts = f.create_group('test')
    ts.create_dataset('a_ff', data=a_ff_test)
    ts.create_dataset('a_ss', data=a_ss_test)

# --- 5. CORRELATION DIAGNOSTIC ---
# Check how well the first 10 sensor modes correlate with the first 10 field modes
print("\n--- Observability Check (Correlation Coeffs) ---")
for i in range(50):
    corr = np.corrcoef(a_ss_train[:, i], a_ff_train[:, i])[0, 1]
    print(f"Mode {i+1} Correlation: {corr:.4f}")

Loading data and extracting strip at x=5.0...
Extracted 242 points at x=5.0.
Computing PODs...

--- Observability Check (Correlation Coeffs) ---
Mode 1 Correlation: -0.7114
Mode 2 Correlation: -0.4009
Mode 3 Correlation: -0.8833
Mode 4 Correlation: -0.0658
Mode 5 Correlation: 0.1161
Mode 6 Correlation: 0.4269
Mode 7 Correlation: -0.0335
Mode 8 Correlation: 0.1403
Mode 9 Correlation: 0.0405
Mode 10 Correlation: -0.0321
Mode 11 Correlation: 0.0578
Mode 12 Correlation: 0.2272
Mode 13 Correlation: 0.0061
Mode 14 Correlation: -0.0356
Mode 15 Correlation: -0.0326
Mode 16 Correlation: 0.0238
Mode 17 Correlation: -0.0915
Mode 18 Correlation: 0.0639
Mode 19 Correlation: 0.1030
Mode 20 Correlation: 0.0237
Mode 21 Correlation: 0.0454
Mode 22 Correlation: -0.0147
Mode 23 Correlation: 0.0171
Mode 24 Correlation: 0.0383
Mode 25 Correlation: 0.0545
Mode 26 Correlation: -0.0121
Mode 27 Correlation: -0.0361
Mode 28 Correlation: -0.1220
Mode 29 Correlation: 0.0857
Mode 30 Correlation: -0.0347
Mode 31 Co